<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Seq2Seq_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%cd drive/MyDrive

/content/drive/MyDrive


In [4]:
dataset_path = "fra.txt"

Remove excess text (attributions) and non-break space chars

In [5]:
def preprocess_text(text):
  text = text.split("CC-BY")[0].strip()
  text = text.replace("\u202f", " ").replace("\xa0", " ")
  no_space = lambda char, prev_char: char in ".,?!" and prev_char != " "
  out = [" " + char if i > 0 and no_space(char, text[i-1]) else char
         for i, char in enumerate(text.lower())]
  result = "".join(out)
  return(result)

Word-level tokenization

In [6]:
def tokenize_data(text, src, tgt):
  parts = text.split("\t")
  if len(parts) == 2:
    src.append([t for t in f"{parts[0]}".split(" ") if t])
    tgt.append([t for t in f"{parts[1]}".split(" ") if t])

In [7]:
src = []
tgt = []

Preprocess and tokenize the data

In [8]:
with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

Pad or truncate the text to obtain a regular shape for computationally efficient operations

In [9]:
def pad_or_truncate(text, num_steps=10):
  if len(text) >= num_steps:
    text = text[:num_steps-1] + ["<eos>"]
  else:
    text += ["<eos>"]
    text += ["<pad>"] * (num_steps - len(text))
  return text


Lookup table

In [10]:
from collections import Counter

def Vocab(src=src, min_freq=2):

  def flatten(src=src):
    return [item for text in src for item in text]

  flattened_seq = flatten()
  counter = Counter(flattened_seq)
  vocab = {"<unk>":0, "<bos>":1}
  i = 2

  for token, freq in counter.items():
    if freq > min_freq and token not in vocab.keys():
      vocab[token] = i
      i += 1
  return vocab

def encode(vocab, word):
  return vocab.get(word, vocab["<unk>"])



In [11]:
src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

In [12]:
tgt[220000]

['<bos>',
 'tom',
 'ferma',
 'les',
 'yeux',
 'et',
 'fit',
 'semblant',
 'de',
 'dormir',
 '<eos>']

In [12]:
lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]


In [13]:
src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

</bos/> is for input fed to the decoder. serves as a signal to the decoder to start producing an output. </eos/> is added to the label as a signal to the decoder to stop outputing tokens.

In [14]:
target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (target_label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)


In [15]:
print(target_label.shape)

torch.Size([240521, 10])


In [16]:
class Embedding():
  def __init__(self, vocab_size, feature_size):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.embedding = torch.randn((vocab_size, feature_size))

  def __call__(self, idx):
    return self.embedding[idx]

  def parameters(self):
    return [self.embedding]

  def __repr__(self):
    return f"Embedding({self.embedding.shape})"

In [ ]:
dense = Embedding(10, 20)
idx = torch.arange(0, 10).view(2, 5)
dense(idx).shape

torch.Size([2, 5, 20])

In [17]:
from sequential_models import Linear, BasicRNN

In [19]:
class RNNEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)

    self.rnn1 = BasicRNN(feature_size, n_neurons)

    #look at hidden state initialization
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.rnn1(dense_input)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    output, h2 = self.rnn2(output1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    return (h1, h2)

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()
    return params

In [20]:
class RNNDecoder():
  def __init__(self, vocab_size, feature_size, n_neurons):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)
    self.rnn1 = BasicRNN(feature_size, n_neurons)
    self.rnn1.W_x = torch.randn((feature_size + n_neurons, n_neurons))
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.linear = Linear(n_neurons, vocab_size)
    self.linear.weights = self.linear.weights * 0.01



  def __call__(self, x, context):
    embedded = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    context = torch.unsqueeze(context, 1)
    # [batch_size, 1, n_hidden]
    context = context.expand((-1, embedded.shape[1], -1))
    # [batch_size, seq_len, feature_size]
    input = torch.cat((embedded, context), -1)
    # [batch_size, seq_len, feature_size + context_features]

    output1, h1 = self.rnn1(input)
    # [seq_len, batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    output2, h2 = self.rnn2(output1)
    # [seq_len, batch_size, n_hidden]
    output2 = output2.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    final_outputs = self.linear(output2)
    # [batch_size, seq_len, linear_hidden]

    return final_outputs


  def __repr__(self):
    rep = f"Decoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons}), \nLinear=({self.neurons, self.vocab_size}))"
    return rep


  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters() + self.linear.parameters()
    return params


In [33]:
def output_mask(valid_len, seq_len=10):
  mask_tensor = []
  for _ in range(valid_len.shape[0]):
    mask_tensor.append(torch.arange(seq_len))

  mask_tensor = torch.stack(mask_tensor)
  valid_len = valid_len.unsqueeze(-1)
  return (mask_tensor < valid_len).type(torch.int32).unsqueeze(-1)





In [130]:
vocab_size_eng, vocab_size_fr = len(eng_vocab.keys()), len(fr_vocab.keys())
feature_size = 1000
n_hidden = 50
encoder = RNNEncoder(vocab_size_eng, feature_size, n_hidden)
decoder = RNNDecoder(vocab_size_fr, feature_size, n_hidden)

In [ ]:
epochs = 1000
params = encoder.parameters() + decoder.parameters()
for _ in range(epochs):
  for p in params:
    p.requires_grad = True

  idxs = torch.randint(0, src_lookup.shape[0]-1, (64, ))
  src_batch = src_lookup[idxs]
  decoder_input_batch = decoder_input[idxs]
  label_batch = target_label[idxs]


  _, h = encoder(src_batch)
  out = decoder(decoder_input_batch, h)

  batch_size, seq_len, v = out.shape
  actual_out = torch.reshape(out, (batch_size * seq_len, v))
  label_batch = label_batch.reshape(batch_size*seq_len).long()

  loss = torch.nn.functional.cross_entropy(actual_out, label_batch, ignore_index=fr_vocab["<pad>"])
  for p in params:
    p.grad = None

  loss.backward()
  lr = 1
  for p in params:
    p.data += -lr * p.grad

  print(loss)



LSTM seq2seq

In [21]:
from sequential_models import LSTMCell

In [22]:
class LSTMEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               batch_size
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons
    self.batch_size = batch_size

    self.embedding = Embedding(vocab_size, feature_size)
    self.lstm1 = LSTMCell(batch_size, feature_size, n_neurons)
    #look at hidden state initialization
    self.lstm2 = LSTMCell(batch_size, n_neurons, n_neurons)


  def __call__(self, x):
    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    _, output1, _, h1 = self.lstm1(dense_input)
    output1 = output1.permute(1, 0, 2)
    _, output2, _, h2 = self.lstm2(output1)
    return (h1, h2)

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nLSTM=({self.feature_size, self.neurons}), \nLSTM=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.lstm1.parameters() + self.lstm2.parameters()
    return params

In [26]:
vocab_size = len(eng_vocab)
feature_size = 100
n_neurons = 50
batch_size = 32
lstm_encoder = LSTMEncoder(vocab_size, feature_size, n_neurons, batch_size)
lstm_encoder

Encoder(
Embedding=torch.Size([9541, 100]),
LSTM=((100, 50)), 
LSTM=((50, 50))
)

In [33]:
epochs = 10
for _ in range(epochs):
  for p in lstm_encoder.parameters():
    p.requires_grad = True

  batch_idx = torch.randint(0, len(src_lookup)-1, (batch_size, ))
  input_batch = src_lookup[batch_idx]
  h1, h2 = lstm_encoder(input_batch)
  print(h1.shape)
  print(h2.shape)

  for p in lstm_encoder.parameters():
    p.grad = None

  break


torch.Size([32, 50])
torch.Size([32, 50])


In [23]:
class LSTMDecoder():
  def __init__(self, vocab_size, feature_size, n_neurons, batch_size):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons
    self.batch_size = batch_size

    self.embedding = Embedding(vocab_size, feature_size)
    self.lstm1 = LSTMCell(batch_size, feature_size + n_neurons, n_neurons)
    self.lstm2 = LSTMCell(batch_size, n_neurons, n_neurons)
    self.linear = Linear(n_neurons, vocab_size)
    self.linear.weights = self.linear.weights * 0.01



  def __call__(self, x, context):
    embedded = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    context = torch.unsqueeze(context, 1)
    # [batch_size, 1, n_hidden_encoder]
    context = context.expand((-1, embedded.shape[1], -1))
    # [batch_size, seq_len, n_hidden_encoder]
    input = torch.cat((embedded, context), -1)
    # [batch_size, seq_len, n_hidden_encoder + feature_size]

    _, output1, _, h1 = self.lstm1(input)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    _, output2, _, h2 = self.lstm2(output1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output2 = output2.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    final_outputs = self.linear(output2)
    # [batch_size, seq_len, linear_hidden]


    return final_outputs


  def __repr__(self):
    rep = f"Decoder(\nEmbedding={self.embedding.embedding.shape},\nLSTM=({self.feature_size, self.neurons}), \nLSTM=({self.neurons, self.neurons}), \nLinear=({self.neurons, self.vocab_size}))"
    return rep


  def parameters(self):
    params = self.embedding.parameters() + self.lstm1.parameters() + self.lstm2.parameters() + self.linear.parameters()
    return params


In [31]:
vocab_size = len(fr_vocab)
feature_size = 100
n_neurons = 50
batch_size = 32
lstm_decoder = LSTMDecoder(vocab_size, feature_size, n_neurons, batch_size)
lstm_decoder

Decoder(
Embedding=torch.Size([15648, 100]),
LSTM=((100, 50)), 
LSTM=((50, 50)), 
Linear=((50, 15648)))

In [35]:
epochs = 10
for _ in range(epochs):
  for p in lstm_decoder.parameters():
    p.requires_grad = True

  batch_idx = torch.randint(0, len(src_lookup)-1, (batch_size, ))
  input_batch = decoder_input[batch_idx]
  h = lstm_decoder(input_batch, h2)
  print(h.shape)

  for p in lstm_decoder.parameters():
    p.grad = None

  break


torch.Size([32, 10, 15648])


In [44]:
vocab_size_eng, vocab_size_fr = len(eng_vocab.keys()), len(fr_vocab.keys())
feature_size = 1000
n_hidden = 100
batch_size = 32
lstm_encoder = LSTMEncoder(vocab_size_eng, feature_size, n_hidden, batch_size)
lstm_decoder = LSTMDecoder(vocab_size_fr, feature_size, n_hidden, batch_size)

In [47]:
epochs = 100
params = lstm_encoder.parameters() + lstm_decoder.parameters()
for _ in range(epochs):
  for p in params:
    p.requires_grad = True

  idxs = torch.randint(0, src_lookup.shape[0]-1, (batch_size, ))
  src_batch = src_lookup[idxs]
  decoder_input_batch = decoder_input[idxs]
  label_batch = target_label[idxs]


  _, h = lstm_encoder(src_batch)
  out = lstm_decoder(decoder_input_batch, h)
  batch_size, seq_len, v = out.shape
  actual_out = torch.reshape(out, (batch_size * seq_len, v))
  label_batch = label_batch.reshape(batch_size*seq_len).long()

  loss = torch.nn.functional.cross_entropy(actual_out, label_batch, ignore_index=fr_vocab["<pad>"])
  for p in params:
    p.grad = None

  loss.backward()
  lr = 1
  for p in params:
    p.data += -lr * p.grad

  print(loss)



tensor(5.5802, grad_fn=<NllLossBackward0>)
tensor(6.1002, grad_fn=<NllLossBackward0>)
tensor(6.2266, grad_fn=<NllLossBackward0>)
tensor(5.8182, grad_fn=<NllLossBackward0>)
tensor(5.6177, grad_fn=<NllLossBackward0>)
tensor(6.0033, grad_fn=<NllLossBackward0>)
tensor(5.6574, grad_fn=<NllLossBackward0>)
tensor(5.8457, grad_fn=<NllLossBackward0>)
tensor(5.5640, grad_fn=<NllLossBackward0>)
tensor(5.5562, grad_fn=<NllLossBackward0>)
tensor(5.7781, grad_fn=<NllLossBackward0>)
tensor(5.8314, grad_fn=<NllLossBackward0>)
tensor(5.6961, grad_fn=<NllLossBackward0>)
tensor(5.8793, grad_fn=<NllLossBackward0>)
tensor(5.7154, grad_fn=<NllLossBackward0>)
tensor(5.4399, grad_fn=<NllLossBackward0>)
tensor(5.7487, grad_fn=<NllLossBackward0>)
tensor(5.6005, grad_fn=<NllLossBackward0>)
tensor(5.6157, grad_fn=<NllLossBackward0>)
tensor(5.8633, grad_fn=<NllLossBackward0>)
tensor(5.7035, grad_fn=<NllLossBackward0>)
tensor(5.7955, grad_fn=<NllLossBackward0>)
tensor(5.6908, grad_fn=<NllLossBackward0>)
tensor(5.42

GRU seq2seq

In [ ]:
from sequential_models import GRUCell

In [ ]:
class GRUEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               batch_size
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons
    self.batch_size = batch_size

    self.embedding = Embedding(vocab_size, feature_size)
    self.gru1 = GRUCell(batch_size, feature_size, n_neurons)
    #look at hidden state initialization
    self.gru2 = GRUCell(batch_size, n_neurons, n_neurons)


  def __call__(self, x):
    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.gru1(dense_input)
    output1 = output1.permute(1, 0, 2)
    output2, h2 = self.gru2(output1)
    return (h1, h2)

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nGRU=({self.feature_size, self.neurons}), \nGRU=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.gru1.parameters() + self.gru2.parameters()
    return params

In [ ]:
vocab_size = len(eng_vocab)
feature_size = 100
n_neurons = 50
batch_size = 32
lstm_encoder = LSTMEncoder(vocab_size, feature_size, n_neurons, batch_size)
lstm_encoder

In [ ]:
epochs = 10
for _ in range(epochs):
  for p in lstm_encoder.parameters():
    p.requires_grad = True

  batch_idx = torch.randint(0, len(src_lookup)-1, (batch_size, ))
  input_batch = src_lookup[batch_idx]
  h1, h2 = lstm_encoder(input_batch)
  print(h1.shape)
  print(h2.shape)

  for p in lstm_encoder.parameters():
    p.grad = None

  break


In [ ]:
class GRUDecoder():
  def __init__(self, vocab_size, feature_size, n_neurons, batch_size):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons
    self.batch_size = batch_size

    self.embedding = Embedding(vocab_size, feature_size)
    self.gru1 = GRUCell(batch_size, feature_size + n_neurons, n_neurons)
    self.gru2 = GRUCell(batch_size, n_neurons, n_neurons)
    self.linear = Linear(n_neurons, vocab_size)
    self.linear.weights = self.linear.weights * 0.01



  def __call__(self, x, context):
    embedded = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    context = torch.unsqueeze(context, 1)
    # [batch_size, 1, n_hidden_encoder]
    context = context.expand((-1, embedded.shape[1], -1))
    # [batch_size, seq_len, n_hidden_encoder]
    input = torch.cat((embedded, context), -1)
    # [batch_size, seq_len, n_hidden_encoder + feature_size]

    output1, h1 = self.gru1(input)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    output2, h2 = self.gru2(output1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output2 = output2.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden]
    final_outputs = self.linear(output2)
    # [batch_size, seq_len, linear_hidden]


    return final_outputs


  def __repr__(self):
    rep = f"Decoder(\nEmbedding={self.embedding.embedding.shape},\nGRU=({self.feature_size, self.neurons}), \nGRU=({self.neurons, self.neurons}), \nLinear=({self.neurons, self.vocab_size}))"
    return rep


  def parameters(self):
    params = self.embedding.parameters() + self.gru1.parameters() + self.gru2.parameters() + self.linear.parameters()
    return params


In [ ]:
vocab_size = len(fr_vocab)
feature_size = 100
n_neurons = 50
batch_size = 32
lstm_decoder = LSTMDecoder(vocab_size, feature_size, n_neurons, batch_size)
lstm_decoder

In [ ]:
epochs = 10
for _ in range(epochs):
  for p in lstm_decoder.parameters():
    p.requires_grad = True

  batch_idx = torch.randint(0, len(src_lookup)-1, (batch_size, ))
  input_batch = decoder_input[batch_idx]
  h = lstm_decoder(input_batch, h2)
  print(h.shape)

  for p in lstm_decoder.parameters():
    p.grad = None

  break


After testing for GRU model, look at the effects of the following on model performance:
1. Sequence length
2. Hidden state size
3. Feature size
4. Minimum frequency
5. Learning rate
6. Weight initialization